In [55]:
import pandas as pd
import re

In [56]:
df = pd.read_csv(
    "../data/guvi_raw_data.csv"
)

In [57]:
print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (45, 4)


,url,title,category,content
0,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,Best Jira: Jira project management Course Onli...
1,https://www.guvi.in,HCL GUVI | Learn to code in your native language,Other,HCL GUVI | Learn to code in your native langua...
2,https://www.guvi.in/referral,Referral | HCL GUVI,Other,Referral | HCL GUVI Home LIVE Classes VLSI Des...
3,https://www.guvi.in/code-kata,HCL GUVI | Learn to code in your native language,Practice,HCL GUVI | Learn to code in your native langua...
4,https://www.guvi.in/zen-class,Zen Class - Career Programs from HCL GUVI,Live Program,Zen Class - Career Programs from HCL GUVI LIVE...


In [58]:
print(df.shape)
print(df.columns)

(45, 4)
Index(['url', 'title', 'category', 'content'], dtype='object')


In [59]:
check_df = df[
    df["title"].str.contains(
        "Explore All GUVI Courses",
        case=False,
        na=False
    )
]

print(
    check_df[
        ["title", "category", "url"]
    ]
)

                                                title category  \
44  Explore All GUVI Courses | Learn Coding in Eng...   Course   

                                        url  
44  https://www.guvi.in/explore-all-courses  


In [60]:
# Check data types and null values

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   url       45 non-null     object
 1   title     45 non-null     object
 2   category  45 non-null     object
 3   content   45 non-null     object
dtypes: object(4)
memory usage: 1.5+ KB


In [61]:
df.isnull().sum()

url         0
title       0
category    0
content     0
dtype: int64

In [62]:
# Check for duplicate rows again

print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [63]:
print("Duplicate URLs:", df["url"].duplicated().sum())

Duplicate URLs: 0


In [64]:
print(
    "Duplicate content:",
    df["content"].duplicated().sum()
)

Duplicate content: 1


In [65]:
# Find the duplicate content

duplicate_content = df[
    df["content"].duplicated(keep=False)
][["title", "category", "url"]]

duplicate_content

,title,category,url
9,HCL GUVI | courses,Course,https://www.guvi.in/courses/tamil/programming/...
32,HCL GUVI | courses,Course,https://www.guvi.in/courses


In [66]:
# Remove duplicate content

df = df.drop_duplicates(
    subset="content",
    keep="first"
).reset_index(drop=True)

In [67]:
print("Dataset Shape:", df.shape)
print("Duplicate content:", df["content"].duplicated().sum())

Dataset Shape: (44, 4)
Duplicate content: 0


### Create a stronger text-cleaning function

In [104]:
def preprocess_text(text):
    # Convert to string
    text = str(text)

    # Replace multiple spaces/newlines with one space
    text = re.sub(r"\s+", " ", text)

    # Remove spaces before punctuation
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)

    # Remove repeated special symbols
    text = re.sub(r"[|]+", " ", text)

    # Remove extra spaces again
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [106]:
df["clean_content"] = df["content"].apply(preprocess_text)

print("Rows:", len(df))
print(
    "Clean content created:",
    df["clean_content"].notna().sum()
)

Rows: 45
Clean content created: 45


In [70]:
print("Original:")
print(df.iloc[0]["content"][:700])

print("\nCleaned:")
print(df.iloc[0]["clean_content"][:700])

Original:
Best Jira: Jira project management Course Online with Certification | GUVI LIVE Classes VLSI Design Programme New IIT Delhi certified AI & Machine Learning New Intel & IIT-M Pravartak certified Data Science IIT-M Pravartak certified Full Stack Development IIT-M Pravartak certified UI/UX Design HCL GUVI certified AI DevOps Program HCL GUVI certified Still Confused? Request a Callback Explore all Programs Courses Practice CodeKata Sharpen your coding skills, prepare for interviews WebKata Build basic frontend & backend development skills SQLKata Master SQL concepts in relational database management FixTheCode A series of programs curated by industry experts IDE - Online Compiler Run & test you

Cleaned:
Best Jira: Jira project management Course Online with Certification GUVI LIVE Classes VLSI Design Programme New IIT Delhi certified AI & Machine Learning New Intel & IIT-M Pravartak certified Data Science IIT-M Pravartak certified Full Stack Development IIT-M Pravartak certified

In [71]:
# Check content lengths

df["clean_length"] = df["clean_content"].str.len()

df["clean_length"].describe()

count       44.000000
mean     12619.909091
std       8626.170815
min       3963.000000
25%       9721.000000
50%      10654.000000
75%      11227.250000
max      48917.000000
Name: clean_length, dtype: float64

In [72]:
print("Shortest page length:", df["clean_length"].min())
print("Longest page length:", df["clean_length"].max())

Shortest page length: 3963
Longest page length: 48917


In [73]:
df["clean_length"].describe()


count       44.000000
mean     12619.909091
std       8626.170815
min       3963.000000
25%       9721.000000
50%      10654.000000
75%      11227.250000
max      48917.000000
Name: clean_length, dtype: float64

In [74]:
# Create a simple chunking function

def split_text_into_chunks(text, chunk_size=1000, overlap=150):
    chunks = []

    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk.strip())

        start = end - overlap

    return chunks

In [75]:
# Test chunking on one page

sample_chunks = split_text_into_chunks(
    df.iloc[0]["clean_content"]
)

print("Number of Chunks:", len(sample_chunks))

Number of Chunks: 14


In [76]:
print("Chunk 1:")
print(sample_chunks[0][:500])

print("\nChunk 2:")
print(sample_chunks[1][:500])

Chunk 1:
Best Jira: Jira project management Course Online with Certification GUVI LIVE Classes VLSI Design Programme New IIT Delhi certified AI & Machine Learning New Intel & IIT-M Pravartak certified Data Science IIT-M Pravartak certified Full Stack Development IIT-M Pravartak certified UI/UX Design HCL GUVI certified AI DevOps Program HCL GUVI certified Still Confused? Request a Callback Explore all Programs Courses Practice CodeKata Sharpen your coding skills, prepare for interviews WebKata Build basi

Chunk 2:
What's New? Resume Builder Community Hub Our Products HackerKID Coding classes platform for K-12 children GUVI For Corporates Meet your Hiring & Training Needs Placement Preparation Ace Up your Aptitude Login Sign up Login Sign up LIVE Classes VLSI Design Programme New IIT Delhi certified AI & Machine Learning New Intel & IIT-M Pravartak certified Data Science IIT-M Pravartak certified Full Stack Development IIT-M Pravartak certified UI/UX Design HCL GUVI certified AI DevOps 

In [77]:
## Improve chunking using words

def split_text_into_chunks(text, chunk_size=200, overlap=30):
    words = text.split()

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk = words[start:end]

        chunks.append(" ".join(chunk))

        start = end - overlap

    return chunks

In [78]:
sample_chunks = split_text_into_chunks(
    df.iloc[0]["clean_content"]
)

print("Number of Chunks:", len(sample_chunks))

Number of Chunks: 10


In [79]:
print("Chunk 1:")
print(sample_chunks[0][:500])

print("\nChunk 2:")
print(sample_chunks[1][:500])

Chunk 1:
Best Jira: Jira project management Course Online with Certification GUVI LIVE Classes VLSI Design Programme New IIT Delhi certified AI & Machine Learning New Intel & IIT-M Pravartak certified Data Science IIT-M Pravartak certified Full Stack Development IIT-M Pravartak certified UI/UX Design HCL GUVI certified AI DevOps Program HCL GUVI certified Still Confused? Request a Callback Explore all Programs Courses Practice CodeKata Sharpen your coding skills, prepare for interviews WebKata Build basi

Chunk 2:
& Machine Learning New Intel & IIT-M Pravartak certified Data Science IIT-M Pravartak certified Full Stack Development IIT-M Pravartak certified UI/UX Design HCL GUVI certified AI DevOps Program HCL GUVI certified Explore all Programs Courses Practice CodeKata Sharpen your coding skills, prepare for interviews WebKata Build basic frontend & backend development skills SQLKata Master SQL concepts in relational database management FixTheCode A series of programs curated by indus

### Apply chunking to the full dataset

In [102]:
chunk_records = []

for _, row in df.iterrows():
    chunks = split_text_into_chunks(
        row["clean_content"]
    )

    for i, chunk in enumerate(chunks):
        chunk_records.append({
            "url": row["url"],
            "title": row["title"],
            "category": row["category"],
            "chunk_id": i,
            "chunk_text": chunk
        })

In [103]:
chunks_df = pd.DataFrame(chunk_records)

In [82]:
print("Total Chunks:", len(chunks_df))
print("Shape:", chunks_df.shape)

chunks_df.head()

Total Chunks: 519
Shape: (519, 5)


,url,title,category,chunk_id,chunk_text
0,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,0,Best Jira: Jira project management Course Onli...
1,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,1,& Machine Learning New Intel & IIT-M Pravartak...
2,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,2,"tech learning is easy, fun, and curated specia..."
3,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,3,Now > WebKata: An interactive platform to mast...
4,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,4,or code and unlock exciting rewards—Amazon vou...


In [83]:
# Check chunk statistics

chunks_df["word_count"] = (
    chunks_df["chunk_text"]
    .str.split()
    .str.len()
)

chunks_df["word_count"].describe()

count    519.000000
mean     190.364162
std       34.134873
min        9.000000
25%      200.000000
50%      200.000000
75%      200.000000
max      200.000000
Name: word_count, dtype: float64

In [84]:
# Find very short chunks

short_chunks = chunks_df[
    chunks_df["word_count"] < 50
]

print("Short Chunks:", len(short_chunks))

short_chunks[
    ["title", "category", "chunk_id", "word_count"]
]

Short Chunks: 12


,title,category,chunk_id,word_count
72,Rewards | HCL GUVI,Other,4,39
95,Leader Board | HCL GUVI,Other,5,9
169,HCL GUVI Campus Ambassador Program | Apply Now,Community,13,12
179,Learn Hub | Free Programming Guides & Tutorial...,Other,9,27
252,Introduction to Agile and Project Management C...,Course,9,34
262,"HCL GUVI For Corporates: Tailored Training, Hi...",Other,9,41
291,FAQ | HCL GUVI,FAQ,9,18
433,Best Python Programming Course with IIT Certif...,Course,10,29
453,Learn Project Management Fundamentals Course O...,Course,9,49
484,Introduction to Dark Web Online Course with Ce...,Course,10,19


In [85]:
# Remove short chunks

chunks_df = chunks_df[
    chunks_df["word_count"] >= 50
].reset_index(drop=True)

In [86]:
print("Final Total Chunks:", len(chunks_df))
print("Minimum Word Count:", chunks_df["word_count"].min())

Final Total Chunks: 507
Minimum Word Count: 56


### Save the processed chunk dataset

In [87]:
chunks_df.to_csv(
    "../data/guvi_chunks.csv",
    index=False
)

print("Chunk dataset saved successfully!")

Chunk dataset saved successfully!


In [88]:
check_chunks = pd.read_csv(
    "../data/guvi_chunks.csv"
)

print("Saved Chunk Shape:", check_chunks.shape)
check_chunks.head()

Saved Chunk Shape: (507, 6)


,url,title,category,chunk_id,chunk_text,word_count
0,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,0,Best Jira: Jira project management Course Onli...,200
1,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,1,& Machine Learning New Intel & IIT-M Pravartak...,200
2,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,2,"tech learning is easy, fun, and curated specia...",200
3,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,3,Now > WebKata: An interactive platform to mast...,200
4,https://www.guvi.in/courses/project-management...,Best Jira: Jira project management Course Onli...,Course,4,or code and unlock exciting rewards—Amazon vou...,200


In [89]:
df = pd.read_csv(
    "../data/guvi_raw_data.csv"
)

print("Updated Raw Dataset Shape:", df.shape)

Updated Raw Dataset Shape: (45, 4)


In [90]:
def preprocess_text(text):
    text = str(text)

    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"[|]+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [91]:
df["clean_content"] = df["content"].apply(preprocess_text)

In [92]:
print("Rows:", len(df))
print("Clean content created:", df["clean_content"].notnull().sum())

Rows: 45
Clean content created: 45


In [93]:
def split_text_into_chunks(text, chunk_size=200, overlap=30):
    words = text.split()

    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk = words[start:end]
        chunks.append(" ".join(chunk))

        start = end - overlap

    return chunks

In [107]:
chunk_records = []

for _, row in df.iterrows():
    chunks = split_text_into_chunks(
        row["clean_content"]
    )

    for i, chunk in enumerate(chunks):
        chunk_records.append({
            "url": row["url"],
            "title": row["title"],
            "category": row["category"],
            "chunk_id": i,
            "chunk_text": chunk
        })

In [108]:
chunks_df = pd.DataFrame(chunk_records)

print("Total Chunks:", len(chunks_df))
print("Shape:", chunks_df.shape)

Total Chunks: 530
Shape: (530, 5)


In [96]:
chunks_df["word_count"] = (
    chunks_df["chunk_text"]
    .str.split()
    .str.len()
)

In [97]:
print("Minimum Word Count:", chunks_df["word_count"].min())
print("Short Chunks:", (chunks_df["word_count"] < 50).sum())

Minimum Word Count: 9
Short Chunks: 12


In [98]:
chunks_df = chunks_df[
    chunks_df["word_count"] >= 50
].reset_index(drop=True)

In [99]:
print("Final Total Chunks:", len(chunks_df))
print("Final Shape:", chunks_df.shape)
print("Minimum Word Count:", chunks_df["word_count"].min())

Final Total Chunks: 518
Final Shape: (518, 6)
Minimum Word Count: 56


In [100]:
chunks_df.to_csv(
    "../data/guvi_chunks.csv",
    index=False
)

print("Updated chunk dataset saved!")
print("Shape:", chunks_df.shape)

Updated chunk dataset saved!
Shape: (518, 6)


In [109]:
chunk_check = chunks_df[
    chunks_df["title"].str.contains(
        "Explore All GUVI Courses",
        case=False,
        na=False
    )
]

print("Matching Chunks:", len(chunk_check))
print()

print(
    chunk_check[
        ["title", "category"]
    ].value_counts()
)

Matching Chunks: 6

title                                                                    category
Explore All GUVI Courses | Learn Coding in English, Hindi, Tamil & More  Course      6
Name: count, dtype: int64


In [110]:
chunks_df["word_count"] = (
    chunks_df["chunk_text"]
    .str.split()
    .str.len()
)

chunks_df = chunks_df[
    chunks_df["word_count"] >= 50
].reset_index(drop=True)

chunks_df.to_csv(
    "../data/guvi_chunks.csv",
    index=False
)

print("Saved corrected chunks successfully!")
print("Final Chunk Shape:", chunks_df.shape)

course_check = chunks_df[
    chunks_df["title"].str.contains(
        "Explore All GUVI Courses",
        case=False,
        na=False
    )
]

print()
print(
    course_check[
        ["title", "category"]
    ].value_counts()
)

Saved corrected chunks successfully!
Final Chunk Shape: (518, 6)

title                                                                    category
Explore All GUVI Courses | Learn Coding in English, Hindi, Tamil & More  Course      5
Name: count, dtype: int64
